In [1]:
import pandas as pd
import math
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr
import faiss
from rank_bm25 import BM25Okapi
import re
import warnings
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from pathlib import Path
import zipfile
import json

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

warnings.filterwarnings('ignore')

# Create common project folders used throughout the notebook.
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# NLTK stop words are helpful for BM25 and word-overlap scoring.
# This keeps the notebook runnable even on a fresh machine.
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()


In [2]:
DEPARTMENT_NAME = "Computer Science and Electrical Engineering"
SOURCE_ROOT = Path("DepCourseInfo")
ZIP_PATH_CANDIDATES = [Path("DepCourseInfo.zip"), Path("/mnt/data/DepCourseInfo.zip")]
PDF_FOLDER_NAME = "downloaded_forms_and_worksheets"

# Prefixes that are common in CS, CE, cybersecurity, networking, data science, AI, and EE/engineering course material.
# Keep this broad because the EECS pages may contain interdisciplinary course documents too.
EECS_PREFIXES = {
    "CAP", "CDA", "CEN", "CGS", "CIS", "CNT", "COP", "COT", "CSC",
    "EEE", "EEL", "EGN", "EGS", "EIN", "EML", "IDC", "ISC", "STA", "MAP", "MAD"
}

# Avoid carrying over biomedical/healthcare-oriented documents from the old topic.
# A CS/EE course with a bioinformatics title is still kept if its prefix is part of EECS_PREFIXES.
BIOMEDICAL_EXCLUDE_TERMS = {
    "bme", "biomedical", "bioengineering", "bio-systems", "stem cell", "tissue engineering"
}

def normalize_whitespace(text):
    return " ".join(str(text).replace("\x00", " ").split())


def clean_title_from_filename(filename):
    stem = Path(filename).stem
    stem = re.sub(r"[_-]+", " ", stem)
    stem = re.sub(r"\s+", " ", stem).strip()
    return stem.title()


def ensure_source_files_available(source_root=SOURCE_ROOT):
    """Use an existing DepCourseInfo folder, or unzip DepCourseInfo.zip if needed."""
    if source_root.exists():
        print(f"Using existing source folder: {source_root.resolve()}")
        return source_root

    zip_path = next((p for p in ZIP_PATH_CANDIDATES if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            "Could not find DepCourseInfo folder or DepCourseInfo.zip. "
            "Place the ZIP next to this notebook or update ZIP_PATH_CANDIDATES."
        )

    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(Path("."))

    if not source_root.exists():
        raise FileNotFoundError(f"Expected extracted folder was not found: {source_root}")

    print(f"Extracted source folder: {source_root.resolve()}")
    return source_root


def guess_document_type(filename, linked_url="", title=""):
    text = f"{filename} {linked_url} {title}".lower()

    if any(word in text for word in ["worksheet", "application", "audit", "form"]):
        return "program_form_or_worksheet"
    if "certificate" in text or "minor" in text or "concentration" in text:
        return "certificate_minor_or_concentration"
    if re.search(r"\b[a-z]{3}\s*[-_ ]?\d{4}", text):
        return "course_syllabus"
    if "syllab" in text:
        return "course_syllabus"
    return "department_document"


def is_relevant_eecs_document(prefix="", title="", filename="", doc_type=""):
    """Return True for CS/EE department material and False for biomedical/healthcare carryover."""
    prefix = str(prefix).upper().strip()
    combined = f"{title} {filename}".lower()

    if prefix:
        return prefix in EECS_PREFIXES

    if any(term in combined for term in BIOMEDICAL_EXCLUDE_TERMS):
        return False

    # Keep program forms, worksheets, certificates, and general department files that do not have a course prefix.
    return True


def extract_pdf_text(pdf_path, max_pages=None):
    """Extract text from a PDF. Returns an empty string for scanned/unreadable PDFs."""
    if PdfReader is None:
        raise ImportError("pypdf is required. Install it with: pip install pypdf")

    try:
        reader = PdfReader(str(pdf_path))
        pages = reader.pages if max_pages is None else reader.pages[:max_pages]
        page_text = []
        for page in pages:
            try:
                page_text.append(page.extract_text() or "")
            except Exception:
                page_text.append("")
        return normalize_whitespace("\n".join(page_text))
    except Exception as exc:
        print(f"Warning: could not read PDF {pdf_path.name}: {exc}")
        return ""


def build_department_corpus(source_root=SOURCE_ROOT, data_dir=DATA_DIR):
    """
    Build a retrieval corpus for CS/EE department information.

    Output columns intentionally include the original template's required columns:
    - doc_id
    - text

    Additional metadata columns are kept for better summaries and inspection.
    """
    source_root = ensure_source_files_available(source_root)
    data_dir.mkdir(exist_ok=True)

    course_csv = source_root / "course_listings.csv"
    pdf_dir = source_root / PDF_FOLDER_NAME

    rows = []
    linked_file_lookup = {}

    if course_csv.exists():
        courses_df = pd.read_csv(course_csv)
        courses_df = courses_df.fillna("")

        # Map downloaded PDF filenames back to their source URLs when possible.
        for _, row in courses_df.iterrows():
            linked = str(row.get("linked_file_or_page", "")).strip()
            if linked.lower().endswith(".pdf"):
                linked_file_lookup[Path(linked).name.lower()] = linked

        # Add one clean course-listing document per unique course/title/source combination.
        course_docs = courses_df.copy()
        course_docs["course_key"] = (
            course_docs["course_code"].astype(str) + " | " +
            course_docs["title_or_context"].astype(str) + " | " +
            course_docs["level"].astype(str)
        )
        course_docs = course_docs.drop_duplicates("course_key")

        for i, row in course_docs.iterrows():
            prefix = str(row.get("prefix", "")).upper().strip()
            course_code = str(row.get("course_code", "")).strip()
            title = str(row.get("title_or_context", "")).strip()

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename="", doc_type="course_listing"):
                continue
            level = str(row.get("level", "")).strip()
            source_page = str(row.get("source_page", "")).strip()
            linked = str(row.get("linked_file_or_page", "")).strip()
            raw_text = str(row.get("raw_text", "")).strip()

            # Keep broad EECS results, but avoid empty/noisy rows.
            if not course_code and not title and not raw_text:
                continue

            doc_id = f"course_{len(rows)+1:04d}_{course_code.replace(' ', '_') or 'unknown'}"
            labeled_text = normalize_whitespace(f"""
Document_Type = course_listing,
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {row.get('number', '')},
Level = {level},
Source_Page = {source_page},
Linked_File_or_Page = {linked},
File_Name = ,
Content = {raw_text if raw_text else f'{course_code} {title}'}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": "course_listing",
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": str(row.get("number", "")).strip(),
                "level": level,
                "source_page": source_page,
                "linked_file_or_page": linked,
                "file_name": "",
                "text": labeled_text,
            })
    else:
        print(f"Warning: course listings file was not found: {course_csv}")

    if pdf_dir.exists():
        pdf_paths = sorted(pdf_dir.rglob("*.pdf"))
        print(f"Reading {len(pdf_paths)} PDFs from {pdf_dir} ...")

        for pdf_path in pdf_paths:
            file_name = pdf_path.name
            linked = linked_file_lookup.get(file_name.lower(), "")
            title = clean_title_from_filename(file_name)
            doc_type = guess_document_type(file_name, linked, title)
            pdf_text = extract_pdf_text(pdf_path)

            # Try to identify course code metadata from the filename or extracted text.
            course_match = re.search(r"\b([A-Z]{3})\s*[-_ ]?(\d{4})\b", f"{file_name} {pdf_text[:500]}", flags=re.IGNORECASE)
            prefix = course_match.group(1).upper() if course_match else ""
            number = course_match.group(2) if course_match else ""
            course_code = f"{prefix} {number}".strip()
            level = "graduate" if number.startswith(("5", "6")) else "undergraduate" if number.startswith(("1", "2", "3", "4")) else ""

            if not is_relevant_eecs_document(prefix=prefix, title=title, filename=file_name, doc_type=doc_type):
                continue

            content = pdf_text if pdf_text else "No extractable PDF text was found. Use the filename and metadata for retrieval."
            doc_id = f"pdf_{len(rows)+1:04d}_{Path(file_name).stem[:60]}"
            labeled_text = normalize_whitespace(f"""
Document_Type = {doc_type},
Title = {title},
Course_Code = {course_code},
Prefix = {prefix},
Number = {number},
Level = {level},
Source_Page = ,
Linked_File_or_Page = {linked},
File_Name = {file_name},
Content = {content}
""")
            rows.append({
                "doc_id": doc_id,
                "document_type": doc_type,
                "title": title,
                "course_code": course_code,
                "prefix": prefix,
                "number": number,
                "level": level,
                "source_page": "",
                "linked_file_or_page": linked,
                "file_name": file_name,
                "text": labeled_text,
            })
    else:
        print(f"Warning: PDF folder was not found: {pdf_dir}")

    corpus_df = pd.DataFrame(rows)
    if corpus_df.empty:
        raise ValueError("No department documents were found. Check SOURCE_ROOT and PDF_FOLDER_NAME.")

    corpus_path = data_dir / "corpus.csv"
    corpus_df.to_csv(corpus_path, index=False)
    print(f"Department corpus saved to: {corpus_path}")
    print(f"Documents created: {len(corpus_df)}")
    print(corpus_df["document_type"].value_counts())
    return corpus_df


def load_corpus(corpus_name="corpus", rebuild=False):
    corpus_fullcsvname = DATA_DIR / f"{corpus_name}.csv"

    if rebuild or not corpus_fullcsvname.exists():
        print("Building a new CS/EE department corpus...")
        corpus_df = build_department_corpus()
    else:
        corpus_df = pd.read_csv(corpus_fullcsvname).fillna("")

    print(f"Corpus: {len(corpus_df)} documents")
    print(f"Columns: {list(corpus_df.columns)}")
    print()
    print("First 3 documents:")
    for _, row in corpus_df.head(3).iterrows():
        preview = str(row["text"])[:220] + "..." if len(str(row["text"])) > 220 else str(row["text"])
        print(f"  {row['doc_id']}: {preview}")
    print()

    return corpus_df


In [3]:
corpus_name = "corpus"

# Use True the first time you switch from the old healthcare corpus to the CS/EE corpus.
# After data/corpus.csv has been created, you can set this to False for faster reruns.
REBUILD_CORPUS = True

corpus_df = load_corpus(corpus_name, rebuild=REBUILD_CORPUS)


Building a new CS/EE department corpus...
Using existing source folder: C:\Users\xande\Projects_Py\EE\DepCourseInfo
Reading 266 PDFs from DepCourseInfo\downloaded_forms_and_worksheets ...
Department corpus saved to: data\corpus.csv
Documents created: 530
document_type
course_listing               275
course_syllabus              173
program_form_or_worksheet     72
department_document           10
Name: count, dtype: int64
Corpus: 530 documents
Columns: ['doc_id', 'document_type', 'title', 'course_code', 'prefix', 'number', 'level', 'source_page', 'linked_file_or_page', 'file_name', 'text']

First 3 documents:
  course_0001_CAP_5615: Document_Type = course_listing, Title = Intro to Neural Networks, Course_Code = CAP 5615, Prefix = CAP, Number = 5615, Level = graduate, Source_Page = https://www.fau.edu/engineering/eecs/research/nrt/faculty/collaborato...
  course_0002_CAP_5615: Document_Type = course_listing, Title = Introduction to Neural Networks, Course_Code = CAP 5615, Prefix = CAP,

In [4]:
DEFAULT_DEPARTMENT_QUERIES = [
    ("q1", "What forms are available for the artificial intelligence certificate?"),
    ("q2", "Which documents describe computer science BS to PhD or combined degree pathways?"),
    ("q3", "Find courses related to data mining, machine learning, and artificial intelligence."),
    ("q4", "What information is available about cybersecurity certificates or security courses?"),
    ("q5", "Which documents mention software engineering requirements, testing, or architecture?"),
    ("q6", "Find electrical engineering graduate syllabi, worksheets, or program information."),
]

def create_default_queries(query_path):
    query_df = pd.DataFrame(DEFAULT_DEPARTMENT_QUERIES, columns=["query_id", "query_text"])
    query_df.to_csv(query_path, index=False)
    print(f"Default CS/EE department queries saved to: {query_path}")
    return query_df


def load_premade_queries(query_names="queries", rebuild=False):
    query_fullcsvnames = DATA_DIR / f"{query_names}.csv"

    if rebuild or not query_fullcsvnames.exists():
        queries_df = create_default_queries(query_fullcsvnames)
    else:
        queries_df = pd.read_csv(query_fullcsvnames)

    print(f"Queries: {len(queries_df)} queries")
    for _, row in queries_df.iterrows():
        print(f"  {row['query_id']}: {row['query_text']}")
    return queries_df


In [5]:
query_names = "queries"

# Use True to replace old healthcare queries with CS/EE department queries.
# After data/queries.csv has been created, you can set this to False for faster reruns.
REBUILD_QUERIES = True

queries_df = load_premade_queries(query_names, rebuild=REBUILD_QUERIES)


Default CS/EE department queries saved to: data\queries.csv
Queries: 6 queries
  q1: What forms are available for the artificial intelligence certificate?
  q2: Which documents describe computer science BS to PhD or combined degree pathways?
  q3: Find courses related to data mining, machine learning, and artificial intelligence.
  q4: What information is available about cybersecurity certificates or security courses?
  q5: Which documents mention software engineering requirements, testing, or architecture?
  q6: Find electrical engineering graduate syllabi, worksheets, or program information.


In [6]:
MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Embedding dimension: 384


In [7]:
def faiss_embedding_process(corpus_df):
    doc_ids = corpus_df['doc_id'].tolist()
    doc_texts = corpus_df['text'].astype(str).tolist()
    
    print(f"Encoding {len(doc_texts)} documents...")
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2 normalize for cosine similarity
    )
    
    print(f"Embedding shape: {doc_embeddings.shape}")
    print(f"Sample vector (first 10 dims): {doc_embeddings[0][:10]}")

    return doc_ids, doc_texts, doc_embeddings
    
    

In [8]:
doc_ids, doc_texts, doc_embeddings = faiss_embedding_process(corpus_df)

Encoding 530 documents...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding shape: (530, 384)
Sample vector (first 10 dims): [-0.08601955 -0.05820242 -0.01651547  0.02603499  0.05125582  0.02605188
 -0.01550748  0.05264784 -0.09018142 -0.02698193]


In [9]:
def faiss_index_process(doc_embeddings):
    dim = doc_embeddings.shape[1]  # 384
    index = faiss.IndexFlatIP(dim)
    index.add(doc_embeddings.astype(np.float32))
    
    print(f"FAISS index built: {index.ntotal} vectors, dimension={dim}")

    return index

In [10]:
index = faiss_index_process(doc_embeddings)

FAISS index built: 530 vectors, dimension=384


In [11]:
def dense_search(query_text, model, index, doc_ids, top_k=10):
    """
    Encode a query and return the top-k most similar documents.

    Returns: list of (doc_id, score, rank) tuples
    """
    # Encode the query with the SAME model and normalization
    query_vec = model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    # Search the FAISS index
    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append((doc_ids[idx], float(score), rank))

    return results

In [12]:
test_query = queries_df.iloc[0]
print(f"Query: {test_query['query_id']} — \"{test_query['query_text']}\"\n")

results = dense_search(test_query['query_text'], model, index, doc_ids, top_k=5)

for doc_id, score, rank in results:
    doc_text = corpus_df[corpus_df['doc_id'] == doc_id]['text'].values[0]
    preview = str(doc_text)[:200] + '...' if len(str(doc_text)) > 200 else str(doc_text)
    print(f"  Rank {rank} | Score: {score:.4f} | {doc_id}")
    print(f"    {preview}\n")

Query: q1 — "What forms are available for the artificial intelligence certificate?"

  Rank 1 | Score: 0.7004 | pdf_0281_artificial-intelligence-certificate-program-application-jan6
    Document_Type = program_form_or_worksheet, Title = Artificial Intelligence Certificate Program Application Jan6, Course_Code = , Prefix = , Number = , Level = , Source_Page = , Linked_File_or_Page = ,...

  Rank 2 | Score: 0.6869 | pdf_0511_professional-ai-certificate-program-application
    Document_Type = program_form_or_worksheet, Title = Professional Ai Certificate Program Application, Course_Code = , Prefix = , Number = , Level = , Source_Page = , Linked_File_or_Page = , File_Name = ...

  Rank 3 | Score: 0.6411 | pdf_0282_artificial-intelligence-minor-program-application-jan6
    Document_Type = program_form_or_worksheet, Title = Artificial Intelligence Minor Program Application Jan6, Course_Code = , Prefix = , Number = , Level = , Source_Page = , Linked_File_or_Page = , File_...

  Rank 4 | Score

In [13]:
TOP_K = 100
all_results = []

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    results = dense_search(qtext, model, index, doc_ids, top_k=TOP_K)
    for doc_id, score, rank in results:
        all_results.append({
            'query_id': qid,
            'doc_id': doc_id,
            'rank': rank,
            'score': round(score, 4),
        })

run_dense_df = pd.DataFrame(all_results)
run_dense_df.to_csv('outputs/run_dense.csv', index=False)

print(f"Dense retrieval results saved: {len(run_dense_df)} rows")
print(f"Queries processed: {run_dense_df['query_id'].nunique()}")
run_dense_df.head(20)

Dense retrieval results saved: 600 rows
Queries processed: 6


,query_id,doc_id,rank,score
0,q1,pdf_0281_artificial-intelligence-certificate-p...,1,0.7004
1,q1,pdf_0511_professional-ai-certificate-program-a...,2,0.6869
2,q1,pdf_0282_artificial-intelligence-minor-program...,3,0.6411
3,q1,pdf_0342_certificate-artificial-intelligence--...,4,0.6172
4,q1,pdf_0343_certificate-artificial-intelligence-p...,5,0.6158
5,q1,course_0005_CAP_5625,6,0.6143
6,q1,course_0024_CAP_6635,7,0.6039
7,q1,pdf_0298_cap4630artificial-intelligence,8,0.6018
8,q1,course_0021_CAP_6629,9,0.5983
9,q1,course_0018_CAP_6619,10,0.5942


In [14]:
def tokenize_for_bm25(text):
    words = re.findall(r"[A-Za-z0-9]+", str(text).lower())
    return [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]


def bm25_tokenize_corpus(corpus_df):
    tokenized_corpus = [tokenize_for_bm25(doc) for doc in corpus_df["text"]]
    return tokenized_corpus


def bm25_corpus(tokenized_corpus):
    bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.8)
    return bm25


In [15]:
tokenized_corpus = bm25_tokenize_corpus(corpus_df)
bm25 = bm25_corpus(tokenized_corpus)

In [16]:
bm25_results = []
for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = tokenize_for_bm25(qrow['query_text'])
    scores = bm25.get_scores(qtext)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_indices, start=1):
        bm25_results.append({
            'query_id': qid,
            'doc_id': doc_ids[idx],
            'rank': rank,
            'score': round(float(scores[idx]), 4),
        })

run_bm25_df = pd.DataFrame(bm25_results)
run_bm25_df.to_csv('outputs/run_bm25.csv', index=False)
print(f"BM25 results: {len(run_bm25_df)} rows")
run_bm25_df.head(10)


BM25 results: 600 rows


,query_id,doc_id,rank,score
0,q1,pdf_0341_certificate-artificial--intelligence-...,1,14.4917
1,q1,pdf_0281_artificial-intelligence-certificate-p...,2,14.2208
2,q1,pdf_0511_professional-ai-certificate-program-a...,3,13.8545
3,q1,pdf_0512_professional-certificate-ai-worksheet,4,13.2579
4,q1,pdf_0282_artificial-intelligence-minor-program...,5,12.3763
5,q1,pdf_0482_minor--artificial-intelligence-applic...,6,12.1148
6,q1,pdf_0278_artificial-inteligence-certificate-wo...,7,12.1106
7,q1,pdf_0509_newcertificate-artificial-intelligenc...,8,11.9859
8,q1,pdf_0279_artificial-inteligence-minor-workshee...,9,11.9776
9,q1,pdf_0280_artificial-intelligence-cert-program-...,10,11.9667


In [17]:
comparison_rows = []

for qid in queries_df['query_id']:
    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    for rank in range(1, 6):
        bm25_row = bm25_q[bm25_q['rank'] == rank]
        dense_row = dense_q[dense_q['rank'] == rank]

        comparison_rows.append({
            'query_id': qid,
            'rank': rank,
            'bm25_doc_id': bm25_row['doc_id'].values[0] if len(bm25_row) > 0 else '',
            'bm25_score': round(bm25_row['score'].values[0], 4) if len(bm25_row) > 0 else '',
            'dense_doc_id': dense_row['doc_id'].values[0] if len(dense_row) > 0 else '',
            'dense_score': round(dense_row['score'].values[0], 4) if len(dense_row) > 0 else '',
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('outputs/comparison.csv', index=False)

print("Side-by-side comparison saved to comparison.csv")
comparison_df.head(10)

Side-by-side comparison saved to comparison.csv


,query_id,rank,bm25_doc_id,bm25_score,dense_doc_id,dense_score
0,q1,1,pdf_0341_certificate-artificial--intelligence-...,14.4917,pdf_0281_artificial-intelligence-certificate-p...,0.7004
1,q1,2,pdf_0281_artificial-intelligence-certificate-p...,14.2208,pdf_0511_professional-ai-certificate-program-a...,0.6869
2,q1,3,pdf_0511_professional-ai-certificate-program-a...,13.8545,pdf_0282_artificial-intelligence-minor-program...,0.6411
3,q1,4,pdf_0512_professional-certificate-ai-worksheet,13.2579,pdf_0342_certificate-artificial-intelligence--...,0.6172
4,q1,5,pdf_0282_artificial-intelligence-minor-program...,12.3763,pdf_0343_certificate-artificial-intelligence-p...,0.6158
5,q2,1,pdf_0293_bsphd-grad-pathway-scholarship-applic...,24.9083,pdf_0277_4-plus-1-itm-csda,0.5043
6,q2,2,pdf_0479_grad-pathway-scholarship-application,18.3410,course_0173_CIS_4213,0.5006
7,q2,3,pdf_0478_grad-pathway-scholarship-application-...,18.2542,course_0186_COP_3014,0.5003
8,q2,4,pdf_0285_bs-phd-computer-science-worksheet12.0...,17.6009,course_0174_CIS_4367,0.5000
9,q2,5,pdf_0284_bs-phd-computer-engineering-worksheet...,16.8373,course_0175_CIS_4634,0.4945


In [18]:
doc_lookup = dict(zip(corpus_df['doc_id'], corpus_df['text'].astype(str)))

for _, qrow in queries_df.iterrows():
    qid = qrow['query_id']
    qtext = qrow['query_text']
    print(f"\n{'='*80}")
    print(f"Query {qid}: \"{qtext}\"")
    print(f"{'='*80}")

    bm25_q = run_bm25_df[run_bm25_df['query_id'] == qid].head(5)
    dense_q = run_dense_df[run_dense_df['query_id'] == qid].head(5)

    print(f"\n{'BM25 Results':<40} | {'Dense Retrieval Results':<40}")
    print(f"{'-'*40} | {'-'*40}")

    for rank in range(5):
        # BM25 side
        if rank < len(bm25_q):
            b = bm25_q.iloc[rank]
            b_text = doc_lookup.get(b['doc_id'], '')[:60]
            bm25_str = f"#{b['rank']} {b['doc_id']} ({b['score']:.2f}) {b_text}"
        else:
            bm25_str = ""

        # Dense side
        if rank < len(dense_q):
            d = dense_q.iloc[rank]
            d_text = doc_lookup.get(d['doc_id'], '')[:60]
            dense_str = f"#{d['rank']} {d['doc_id']} ({d['score']:.4f}) {d_text}"
        else:
            dense_str = ""

        print(f"{bm25_str:<40} | {dense_str:<40}")

    # Overlap analysis
    bm25_docs = set(bm25_q['doc_id'])
    dense_docs = set(dense_q['doc_id'])
    overlap = bm25_docs & dense_docs
    print(f"\nOverlap: {len(overlap)} / 5 docs appear in both top-5")
    if overlap:
        print(f"  Shared docs: {overlap}")
    print(f"  BM25-only: {bm25_docs - dense_docs}")
    print(f"  Dense-only: {dense_docs - bm25_docs}")


Query q1: "What forms are available for the artificial intelligence certificate?"

BM25 Results                             | Dense Retrieval Results                 
---------------------------------------- | ----------------------------------------
#1 pdf_0341_certificate-artificial--intelligence-program-application (14.49) Document_Type = program_form_or_worksheet, Title = Certifica | #1 pdf_0281_artificial-intelligence-certificate-program-application-jan6 (0.7004) Document_Type = program_form_or_worksheet, Title = Artificia
#2 pdf_0281_artificial-intelligence-certificate-program-application-jan6 (14.22) Document_Type = program_form_or_worksheet, Title = Artificia | #2 pdf_0511_professional-ai-certificate-program-application (0.6869) Document_Type = program_form_or_worksheet, Title = Professio
#3 pdf_0511_professional-ai-certificate-program-application (13.85) Document_Type = program_form_or_worksheet, Title = Professio | #3 pdf_0282_artificial-intelligence-minor-program-applicatio

In [19]:
#bm25_weight= 2.5, dense_weight= 0.3
def rrf_process(bm25_df, dense_df, k=60, bm25_weight= 2.6, dense_weight= 0.25):
    fused_results = []

    bm_ids = set(bm25_df["query_id"])
    dense_ids = set(dense_df["query_id"])
    query_ids = sorted(bm_ids.union(dense_ids))

    for qid in query_ids:
        rrf_scores = {}

        bm25_q = bm25_df[bm25_df["query_id"] == qid]
        dense_q = dense_df[dense_df["query_id"] == qid]

        # BM25 contribution
        for _, row in bm25_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                bm25_weight / (k + rank)
            )

        # Dense contribution
        for _, row in dense_q.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (
                dense_weight / (k + rank)
            )

        ranked_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        for final_rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            fused_results.append({
                "query_id": qid,
                "doc_id": doc_id,
                "rrf_score": round(score, 6),
                "rank": final_rank
            })

    fused_df = pd.DataFrame(fused_results)
    return fused_df

    

In [20]:
def preprocess_text(text):
    words = re.findall(r"[A-Za-z0-9]+", str(text).lower())
    
    cleaned_words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 1
    ]
    
    return set(cleaned_words)


def simple_word_match_percent(query_text, doc_text):
    query_words = preprocess_text(query_text)
    doc_words = preprocess_text(doc_text)

    if not query_words:
        return 0.0

    overlap = query_words.intersection(doc_words)
    return (len(overlap) / len(query_words)) * 100


def semantic_similarity_score(query_text, doc_text, embedding_model):
    embeddings = embedding_model.encode(
        [str(query_text), str(doc_text)],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    query_vec = embeddings[0]
    doc_vec = embeddings[1]

    cosine_sim = float(np.dot(query_vec, doc_vec))
    cs = round(cosine_sim, 2)
    return cs


In [21]:
SUMMARY_MODEL_NAME = "google/flan-t5-small"

print(f"Loading summarization model: {SUMMARY_MODEL_NAME}")
summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARY_MODEL_NAME)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARY_MODEL_NAME)
print("Summarization model loaded.")

# Lookup tables for doc_id -> full document text and metadata
corpus_df = corpus_df.fillna("")
doc_lookup = dict(zip(corpus_df["doc_id"], corpus_df["text"].astype(str)))
metadata_lookup = corpus_df.set_index("doc_id").to_dict(orient="index")

FIELD_NAMES = [
    "Document_Type",
    "Title",
    "Course_Code",
    "Prefix",
    "Number",
    "Level",
    "Source_Page",
    "Linked_File_or_Page",
    "File_Name",
    "Content"
]


def field_to_pattern(field_name):
    """
    Allow matching labels written with underscores or spaces.
    Example: Course_Code or Course Code
    """
    return re.escape(field_name).replace(r"\_", r"[ _]+")


def extract_structured_fields(doc_text):
    """
    Extract the labeled metadata fields from one department corpus document.
    """
    text = " ".join(str(doc_text).replace("\n", " ").split())
    extracted = {}

    for i, field in enumerate(FIELD_NAMES):
        current_label = field_to_pattern(field)

        if i < len(FIELD_NAMES) - 1:
            next_labels = [field_to_pattern(f) for f in FIELD_NAMES[i + 1:]]
            next_label_pattern = "|".join(next_labels)
            pattern = rf"{current_label}\s*=\s*(.*?)(?=,\s*(?:{next_label_pattern})\s*=|$)"
        else:
            pattern = rf"{current_label}\s*=\s*(.*)$"

        match = re.search(pattern, text, flags=re.IGNORECASE)
        extracted[field] = match.group(1).strip(" ,") if match else "Not found"

    return extracted


def summarize_document(query_text, doc_text, tokenizer, summary_model, embedding_model, doc_id=None):
    doc_text = str(doc_text).strip()

    empty_result = {
        "Document Type": "Not found",
        "Title": "Not found",
        "Course Code": "Not found",
        "Level": "Not found",
        "File Name": "Not found",
        "Source": "Not found",
        "Word Match %": 0.0,
        "Semantic Similarity": 0.0,
        "Content Summary": "No department document content available."
    }

    if not doc_text:
        return empty_result

    extracted = extract_structured_fields(doc_text)
    content_text = extracted.get("Content", "")
    if not content_text or content_text == "Not found":
        content_text = doc_text

    word_match = simple_word_match_percent(query_text, doc_text)
    semantic_sim = semantic_similarity_score(query_text, doc_text, embedding_model)

    prompt = f"""
Summarize this Computer Science and Electrical Engineering department document in 1 to 2 sentences.

Requirements:
- Focus on forms, worksheets, certificates, degree requirements, course syllabi, prerequisites, course topics, or program information.
- Explain why the document may be relevant to the query.
- Do not use healthcare or patient language.

Metadata:
Document Type = {extracted['Document_Type']}
Title = {extracted['Title']}
Course Code = {extracted['Course_Code']}
Level = {extracted['Level']}
File Name = {extracted['File_Name']}

Query:
{query_text}

Document content:
{content_text[:2500]}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = summary_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=False
    )

    content_summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    if not content_summary:
        content_summary = "No concise summary was generated for this document."

    source_value = extracted["Linked_File_or_Page"]
    if source_value == "Not found" or not source_value:
        source_value = extracted["Source_Page"]

    return {
        "Document Type": extracted["Document_Type"],
        "Title": extracted["Title"],
        "Course Code": extracted["Course_Code"],
        "Level": extracted["Level"],
        "File Name": extracted["File_Name"],
        "Source": source_value if source_value else "Not found",
        "Word Match %": round(word_match, 2),
        "Semantic Similarity": round(semantic_sim, 4),
        "Content Summary": content_summary
    }


Loading summarization model: google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Summarization model loaded.


In [22]:
run_rrf_df = rrf_process(run_bm25_df, run_dense_df, k=60)

# Save fused results
run_rrf_df.to_csv("outputs/run_rrf.csv", index=False)

# Print top 5 per query with summaries
for qid in run_rrf_df["query_id"].unique():
    print(f"\n{'='*80}")
    print(f"RRF Results for Query {qid}")
    print(f"{'='*80}")

    query_matches = queries_df.loc[queries_df["query_id"] == qid, "query_text"]
    if query_matches.empty:
        print(f"No query text found for {qid}")
        continue

    query_text = query_matches.iloc[0]
    top_5 = run_rrf_df[run_rrf_df["query_id"] == qid].head(5)

    for _, row in top_5.iterrows():
        doc_id = row["doc_id"]
        doc_text = doc_lookup.get(doc_id, "")

        doc_summary = summarize_document(
            query_text=query_text,
            doc_text=doc_text,
            tokenizer=summary_tokenizer,
            summary_model=summary_model,
            embedding_model=model,
            doc_id=doc_id
        )

        print(
            f"Rank #{row['rank']:>2} | "
            f"Doc: {doc_id} | "
            f"RRF Score: {row['rrf_score']:.6f}"
        )
        print(f"Title: {doc_summary['Title']}")
        print(f"Type: {doc_summary['Document Type']} | Course: {doc_summary['Course Code']} | Level: {doc_summary['Level']}")
        print(f"Word Match: {doc_summary['Word Match %']}% | Semantic Similarity: {doc_summary['Semantic Similarity']}")
        print(f"Summary: {doc_summary['Content Summary']}")
        print("-" * 80)



RRF Results for Query q1
Rank # 1 | Doc: pdf_0281_artificial-intelligence-certificate-program-application-jan6 | RRF Score: 0.046034
Title: Artificial Intelligence Certificate Program Application Jan6
Type: program_form_or_worksheet | Course:  | Level: 
Word Match: 80.0% | Semantic Similarity: 0.7
Summary: Applicants must complete the application form and submit the application form.
--------------------------------------------------------------------------------
Rank # 2 | Doc: pdf_0341_certificate-artificial--intelligence-program-application | RRF Score: 0.045311
Title: Certificate Artificial Intelligence Program Application
Type: program_form_or_worksheet | Course:  | Level: 
Word Match: 80.0% | Semantic Similarity: 0.52
Summary: Applicants for the Artificial Intelligence Certificate are required to complete the application.
--------------------------------------------------------------------------------
Rank # 3 | Doc: pdf_0511_professional-ai-certificate-program-application | RRF

In [23]:
qrels_path = DATA_DIR / "qrels.csv"
qrels_available = qrels_path.exists()

if qrels_available:
    qrels_df = pd.read_csv(qrels_path)
else:
    qrels_df = pd.DataFrame(columns=["query_id", "doc_id", "relevance"])
    print("No data/qrels.csv file was found. Skipping supervised retrieval evaluation metrics.")
    print("Retrieval still works; qrels are only needed for P@K, MAP, and nDCG scoring.")


def precision_at_k(ranked_doc_ids, rel_map, k=10):
    top = ranked_doc_ids[:k]
    if len(top) == 0:
        return 0.0
    hits = sum(1 for d in top if rel_map.get(d, 0) > 0)  # relevance 1 or 2 counts as relevant
    return hits / len(top)

# Build relevance lookup: qid -> {doc_id: relevance}
rel_by_q = {}
if qrels_available:
    for qid, g in qrels_df.groupby("query_id"):
        rel_by_q[str(qid)] = {
            str(r.doc_id): int(r.relevance)
            for r in g.itertuples(index=False)
        }


def evaluate_p10(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping P@{k} for {run_name}; qrels are not available.")
        return 0.0

    p10_values = []

    print(f"\nP@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        p10 = precision_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        p10_values.append(p10)
        print(f"{qid}: {p10:.3f}")

    mean_p10 = float(np.mean(p10_values)) if p10_values else 0.0
    print(f"Mean P@{k} for {run_name}: {mean_p10:.3f}")
    return mean_p10

# Evaluate BM25
mean_p10_bm25 = evaluate_p10(run_bm25_df, "BM25", k=10)

# Evaluate FAISS / Dense retrieval
mean_p10_faiss = evaluate_p10(run_dense_df, "FAISS Dense Search", k=10)

# Evaluate RRF
mean_p10_rrf = evaluate_p10(run_rrf_df, "RRF Fusion", k=10)



P@10 per query (BM25):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000
Mean P@10 for BM25: 0.000

P@10 per query (FAISS Dense Search):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000
Mean P@10 for FAISS Dense Search: 0.000

P@10 per query (RRF Fusion):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000
Mean P@10 for RRF Fusion: 0.000


In [24]:
print("\n" + "=" * 50)
print("Mean P@10 Summary")
print("=" * 50)
print(f"BM25:              {mean_p10_bm25:.3f}")
print(f"FAISS Dense:       {mean_p10_faiss:.3f}")
print(f"RRF Fusion:        {mean_p10_rrf:.3f}")


Mean P@10 Summary
BM25:              0.000
FAISS Dense:       0.000
RRF Fusion:        0.000


In [25]:
def average_precision(ranked_doc_ids, rel_map):
    # AP uses binary relevance: treat relevance 1 or 2 as relevant
    total_relevant = sum(1 for v in rel_map.values() if v > 0)
    if total_relevant == 0:
        return 0.0

    hits = 0
    ap_sum = 0.0

    for i, d in enumerate(ranked_doc_ids, start=1):
        if rel_map.get(d, 0) > 0:
            hits += 1
            ap_sum += hits / i

    return ap_sum / total_relevant


def evaluate_map(run_df, run_name):
    if not qrels_available:
        print(f"Skipping MAP for {run_name}; qrels are not available.")
        return 0.0

    ap_values = []

    print(f"\nAverage Precision (AP) per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ap = average_precision(ranked, rel_by_q.get(str(qid), {}))
        ap_values.append(ap)
        print(f"{qid}: {ap:.3f}")

    map_score = float(np.mean(ap_values)) if len(ap_values) > 0 else 0.0
    print(f"\nMAP ({run_name}): {map_score:.3f}")
    return map_score


# ---- BM25 ----
map_bm25 = evaluate_map(run_bm25_df, "BM25")

# ---- FAISS / Dense ----
map_faiss = evaluate_map(run_dense_df, "FAISS Dense Search")

# ---- RRF ----
map_rrf = evaluate_map(run_rrf_df, "RRF Fusion")


print("\n" + "=" * 50)
print("MAP Summary")
print("=" * 50)
print(f"BM25:         {map_bm25:.3f}")
print(f"FAISS Dense:  {map_faiss:.3f}")
print(f"RRF Fusion:   {map_rrf:.3f}")



Average Precision (AP) per query (BM25):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

MAP (BM25): 0.000

Average Precision (AP) per query (FAISS Dense Search):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

MAP (FAISS Dense Search): 0.000

Average Precision (AP) per query (RRF Fusion):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

MAP (RRF Fusion): 0.000

MAP Summary
BM25:         0.000
FAISS Dense:  0.000
RRF Fusion:   0.000


In [26]:
def dcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = 0.0
    for i, d in enumerate(ranked_doc_ids[:k], start=1):
        gain = rel_map.get(d, 0)  # graded relevance: 0, 1, 2
        dcg += gain / math.log2(i + 1)
    return dcg


def ndcg_at_k(ranked_doc_ids, rel_map, k=10):
    dcg = dcg_at_k(ranked_doc_ids, rel_map, k)

    # Ideal DCG: sort true relevance scores from highest to lowest
    ideal_gains = sorted(rel_map.values(), reverse=True)
    idcg = 0.0
    for i, gain in enumerate(ideal_gains[:k], start=1):
        idcg += gain / math.log2(i + 1)

    return (dcg / idcg) if idcg > 0 else 0.0


def evaluate_ndcg(run_df, run_name, k=10):
    if not qrels_available:
        print(f"Skipping nDCG@{k} for {run_name}; qrels are not available.")
        return 0.0

    ndcg_values = []

    print(f"\nnDCG@{k} per query ({run_name}):")
    for qid, g in run_df.groupby("query_id"):
        ranked = g.sort_values("rank")["doc_id"].astype(str).tolist()
        ndcg = ndcg_at_k(ranked, rel_by_q.get(str(qid), {}), k=k)
        ndcg_values.append(ndcg)
        print(f"{qid}: {ndcg:.3f}")

    mean_ndcg = float(np.mean(ndcg_values)) if len(ndcg_values) > 0 else 0.0
    print(f"\nMean nDCG@{k} ({run_name}): {mean_ndcg:.3f}")
    return mean_ndcg

# ---- BM25 ----
mean_ndcg_bm25 = evaluate_ndcg(run_bm25_df, "BM25", k=30)

# ---- FAISS / Dense ----
mean_ndcg_faiss = evaluate_ndcg(run_dense_df, "FAISS Dense Search", k=30)

# ---- RRF ----
mean_ndcg_rrf = evaluate_ndcg(run_rrf_df, "RRF Fusion", k=30)

print("\n" + "=" * 50)
print("nDCG@30 Summary")
print("=" * 50)
print(f"BM25:         {mean_ndcg_bm25:.3f}")
print(f"FAISS Dense:  {mean_ndcg_faiss:.3f}")
print(f"RRF Fusion:   {mean_ndcg_rrf:.3f}")



nDCG@30 per query (BM25):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

Mean nDCG@30 (BM25): 0.000

nDCG@30 per query (FAISS Dense Search):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

Mean nDCG@30 (FAISS Dense Search): 0.000

nDCG@30 per query (RRF Fusion):
q1: 0.000
q2: 0.000
q3: 0.000
q4: 0.000
q5: 0.000
q6: 0.000

Mean nDCG@30 (RRF Fusion): 0.000

nDCG@30 Summary
BM25:         0.000
FAISS Dense:  0.000
RRF Fusion:   0.000
